# SafeTune — RECOVER Demo

Weight-space safety recovery, end to end. A finished/drifted model is repaired by **editing its weights directly** — no training.

`recover.apply_resta` computes the safety vector `v = θ_aligned − θ_base` and applies `θ_target += α·v`. The contract is uniform across every recover method: `apply_X(target=, base=, aligned=)`.

To stay self-contained and fast this demo **synthesizes** the three checkpoints from one small model (a random "alignment" delta and a random "drift" delta). In real use, `base` / `aligned` / `target` are three genuine checkpoints; on real drifted Llamas `apply_resta` reaches +7.9pp safety.

> Mirrors `examples/recover_quickstart.py`.

## Install SafeTune

In [1]:
!pip install safetune -q   # from PyPI


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


## Setup + synthesize checkpoints

In [2]:
import copy, torch
from transformers import AutoModelForCausalLM
from safetune.recover import apply_resta

MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device = {device}")

base = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float32).to(device)

# aligned = base + small "alignment" delta ; drifted = base + "drift" delta
torch.manual_seed(0)
aligned = copy.deepcopy(base)
drifted = copy.deepcopy(base)
with torch.no_grad():
    for (_, pb), (_, pa), (_, pd) in zip(
        base.named_parameters(), aligned.named_parameters(), drifted.named_parameters()
    ):
        if pb.dim() >= 2:  # perturb weight matrices only
            pa.add_(torch.randn_like(pa) * 0.01)   # alignment signal
            pd.add_(torch.randn_like(pd) * 0.02)   # drift away from safety

# snapshot drifted weights to measure how much recovery moved them
before = {n: p.detach().clone() for n, p in drifted.named_parameters()}
print("checkpoints synthesized: base, aligned, drifted")

device = cuda


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

checkpoints synthesized: base, aligned, drifted


## Apply `apply_resta`

In [3]:
patched = apply_resta(finetuned=drifted, base=base, aligned=aligned, alpha=1.0)

changed, total_delta, max_delta = 0, 0.0, 0.0
for n, p in patched.named_parameters():
    d = (p.detach() - before[n]).float()
    norm = d.norm().item()
    if norm > 1e-9:
        changed += 1
        total_delta += norm
        max_delta = max(max_delta, d.abs().max().item())

print(f"apply_resta edited {changed} parameter tensors in place.")
print(f"total ‖Δθ‖ = {total_delta:.4f}   max |Δθ| = {max_delta:.4f}")
print("(the safety vector θ_aligned − θ_base, added onto the drifted model)")

apply_resta edited 169 parameter tensors in place.
total ‖Δθ‖ = 2212.1664   max |Δθ| = 0.0591
(the safety vector θ_aligned − θ_base, added onto the drifted model)


## What just happened

`apply_resta` performed a **training-free weight edit** — the safety task vector `θ_aligned − θ_base` was added onto the drifted model, pulling it back toward the aligned checkpoint. No optimizer steps, no forget set.

Every `recover` method shares this uniform contract:
```python
patched = safetune.recover.apply_resta(finetuned=..., base=..., aligned=...)
```

See `docs/getting-started/index.md` for the full recover catalog (whole-model → subspace → layer → neuron → circuit).

**Next:** explore the other classes:
- Steer: [`steer_demo.ipynb`](steer_demo.ipynb)
- Harden: [`harden_demo.ipynb`](harden_demo.ipynb)
- Unlearn: [`unlearn_demo.ipynb`](unlearn_demo.ipynb)